# Task 1.1 — Core Contribution / Architecture

**Paper:** *Dual Coordinate Descent Methods for Logistic Regression and Maximum Entropy Models*  
**Authors:** Hsiang-Fu Yu, Fang-Lan Huang, Chih-Jen Lin  
**Venue:** Machine Learning (Springer), Vol. 85, pp. 41–75, 2011  
**DOI:** 10.1007/s10994-010-5221-8

---

## Step-by-Step Method Description

### Step 1: Formulate the Dual Problem for Logistic Regression
- **Description:** The standard primal logistic regression minimises the regularised negative log-likelihood (Eq. 1): $P_{LR}(w) = C \sum_{i=1}^{l} \log(1 + e^{-y_i w^T x_i}) + \frac{1}{2}w^T w$. The authors derive an equivalent *dual* problem (Eq. 3) in terms of dual variables $\alpha \in \mathbb{R}^l$ where the dual objective is $D_{LR}(\alpha) = \frac{1}{2}\alpha^T Q\alpha + \sum_i [\alpha_i \log \alpha_i + (C-\alpha_i)\log(C-\alpha_i)]$, subject to $0 < \alpha_i < C$.
- **Reference:** Section 1, Equations (1) and (3).
- **Purpose:** Reformulating as a dual problem enables coordinate descent to update one variable at a time at lower cost per step, and lets the primal weight vector $w(\alpha) = \sum_i y_i \alpha_i x_i$ be maintained cheaply.

---

### Step 2: Establish Interior-Point Nature of the Optimal Solution (Theorem 1)
- **Description:** Unlike SVM, where the dual optimum is often sparse (many $\alpha_i = 0$), the authors prove (Theorem 1, Appendix A.2) that the LR dual optimum $\alpha^* \in (0, C)^l$ — strictly in the open interval. This is established using a convexity argument and the boundary behaviour of the log terms.
- **Reference:** Theorem 1, Section 3.1, Appendix A.2.
- **Purpose:** This result rules out initialising at $\alpha = 0$ (as done for SVM) and motivates the special initialisation strategy in Step 3.

---

### Step 3: Initialise $\alpha$ Safely Near Zero
- **Description:** Since $\alpha = 0$ is not feasible, the paper uses $\alpha_i = \min(\epsilon_1 C, \epsilon_2)$ for small constants $\epsilon_1 = 10^{-3}$ and $\epsilon_2 = 10^{-8}$ (Eq. 36). The corresponding weight vector is initialised as $w \leftarrow \sum_i \alpha_i y_i x_i$. The authors justify this by observing that correctly classified instances tend to have small $\alpha_i$, mirroring the sparsity intuition from SVM.
- **Reference:** Equation (36), Section 3.4.
- **Purpose:** Provides a valid, computationally stable starting point that is close to the optimal region for typical datasets.

---

### Step 4: Maintain the Primal Weight Vector $w(\alpha)$
- **Description:** To avoid the $O(nl)$ cost of computing $(Q\alpha)_i = \sum_j Q_{ij}\alpha_j$ from scratch, the paper maintains the auxiliary vector $w(\alpha) = \sum_i y_i \alpha_i x_i$ (Eq. 22). The gradient component needed for the sub-problem is then computed as $(Q\alpha)_i = y_i w(\alpha)^T x_i$ in $O(n)$ time. After each update of $\alpha_i$ by $z$, $w$ is updated via $w \leftarrow w + z y_i x_i$ (Eq. 24).
- **Reference:** Equations (22)–(24), Section 3.1.
- **Purpose:** Reduces the dominant per-iteration cost from $O(nl)$ to $O(n)$, making the method scalable to large datasets.

---

### Step 5: Construct the One-Variable Sub-Problem
- **Description:** At each iteration, one index $i$ is selected (via random permutation of $\{1,\ldots,l\}$). The sub-problem over the scalar increment $z$ is (Eq. 18): $\min_z \; (c_1+z)\log(c_1+z) + (c_2-z)\log(c_2-z) + \frac{a}{2}z^2 + bz$, subject to $-c_1 \le z \le c_2$, where $c_1=\alpha_i$, $c_2=C-\alpha_i$, $a=Q_{ii}=x_i^T x_i$, $b=y_i w^T x_i$. This sub-problem has no closed-form solution, unlike SVM.
- **Reference:** Equation (18), Section 3.1.
- **Purpose:** Isolates a one-dimensional optimisation that captures the curvature of the LR dual for a single variable.

---

### Step 6: Solve the Sub-Problem via Modified Newton Method (Algorithm 4)
- **Description:** Because the log terms in the sub-problem introduce numerical hazards (catastrophic cancellation when $z \approx -c_1$ or $z \approx c_2$), the paper introduces reformulations $g_1(Z_1)$ and $g_2(Z_2)$ (Section 3.3, Eq. 32). It chooses which formulation to use based on whether the optimum is closer to the lower or upper bound (Eq. 33–34). Newton updates are applied: $z^{k+1} = z^k - g_t'(Z_t^k)/g_t''(Z_t^k)$. If the update overshoots the interval boundary, the paper applies the contraction rule $Z_t^{k+1} = \xi Z_t^k$ (with $\xi \in (0,1)$) instead of a line search (Eq. 27). Global convergence is proven (Theorems 3–5) without any line search.
- **Reference:** Algorithm 4, Equations (27)–(35), Section 3.2–3.3.
- **Purpose:** Provides a numerically robust, line-search-free Newton solver for the sub-problem — the central algorithmic contribution of the paper.

---

### Step 7: Update $\alpha_i$ and $w(\alpha)$; Repeat
- **Description:** After solving the sub-problem, the algorithm updates $\alpha_i \leftarrow Z_1^*$, $\alpha_i' \leftarrow Z_2^* = C - Z_1^*$, and $w \leftarrow w + (Z_1^* - \alpha_i^{\text{old}}) y_i x_i$ (Algorithm 5, Step 3–4). The outer loop cycles through all $l$ indices (with random permutation) until the KKT conditions are met. The paper proves linear convergence of this outer loop (Theorem 6, Appendix A.6) using the framework of Luo and Tseng (1992).
- **Reference:** Algorithm 5, Theorem 6, Section 3.4.
- **Purpose:** Completes the iterative optimisation loop, returning the optimal dual solution $\alpha^*$ and, via $w(\alpha^*)$, the optimal primal weight vector.

---

## Final Summary Sentence

This paper solves the problem of *efficiently training logistic regression and maximum entropy models at scale* by adapting coordinate descent from the SVM dual to the LR/ME dual — an adaptation that is non-trivial because the LR dual objective contains logarithmic terms requiring a novel numerically stable, line-search-free modified Newton sub-solver — and the authors demonstrate that this approach is faster than state-of-the-art primal solvers (TRON, L-BFGS, CDprimal) on large NLP datasets.

In [2]:
# No code required for Task 1.1 — all responses are in markdown cells as instructed.
print("Task 1.1: Core Contribution / Architecture — markdown responses above.")

Task 1.1: Core Contribution / Architecture — markdown responses above.
